# Olympic Medals: Will an athlete win a medal at the NEXT Olympics?

**What this notebook does (in order):**
1. EDA  2. Preprocessing  3. Visualization  4. Distributions  5. Sampling  6. Hypothesis testing  7. Correlation & covariance  8. Models (simple linear, multiple linear, ridge, lasso, decision tree, random forest)  9. Pick the best model  10. Accuracy & overfitting check  11. Save a `.pkl` file

**Data:** `Medals.csv` has one row per athlete per Olympics (2000-2012, Summer and Winter). **Every row is a medal winner**, so there are no "did not win" examples in the file.

**Target we predict:** *Did the athlete win a medal again at the next Games of the same season (4 years later)?*  (`1` = yes, `0` = no).  
The model outputs a **probability (0-100%)** that the athlete medals again.

**Run it:** Runtime > Run all. When asked, upload `Medals.csv`.

## 0. Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (accuracy_score, roc_auc_score, mean_squared_error,
                             mean_absolute_error, r2_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_style("whitegrid")

## 1. Load the data

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()              # choose Medals.csv
    path = list(uploaded.keys())[0]
except ImportError:                        # running outside Colab
    path = "Medals.csv"

df_raw = pd.read_csv(path, encoding="cp1252")
df_raw.head()

## 2. EDA (Exploratory Data Analysis)

In [ ]:
print("Shape:", df_raw.shape)
print()
df_raw.info()

In [ ]:
print("Missing values:")
print(df_raw.isnull().sum())
print()
print("Duplicate rows:", df_raw.duplicated().sum())
print()
print(df_raw.describe().round(2))

In [ ]:
print("Rows per Olympic year:")
print(df_raw["Year"].value_counts().sort_index())
print()
print("Unique athletes:", df_raw["Athlete"].nunique())
print("Every row has at least one medal:", (df_raw["Total Medals"] >= 1).all())

## 3. Preprocessing

Steps:
- remove rows with missing athlete name or age, and exact duplicates
- keep **one row per athlete per Olympics** (add up medals if the name repeats)
- `Season`: 1 = Summer (years divisible by 4), 0 = Winter
- **History features** (only use the PAST, so there is no cheating):
  - `Prev_Appearances` = how many earlier Games (same season) the athlete already medalled at
  - `Prev_Medals` = medals won at those earlier Games
- `Gold_Share` = fraction of this Games' medals that were gold
- **Label** = 1 if the same athlete appears again 4 years later, else 0.  
  The file ends in 2012, so only Games up to 2008 can get a label. 2010 and 2012 rows are kept aside for "future" predictions.

In [ ]:
df = df_raw.copy()
df.columns = ["Athlete", "Age", "Year", "Closing_Date", "Gold", "Silver", "Bronze", "Total"]

df = df.dropna(subset=["Athlete", "Age"])
df = df.drop_duplicates()

# one row per athlete per Olympics
df = df.groupby(["Athlete", "Year"], as_index=False).agg(
    Age=("Age", "max"), Gold=("Gold", "sum"), Silver=("Silver", "sum"),
    Bronze=("Bronze", "sum"), Total=("Total", "sum"))

df["Season"] = np.where(df["Year"] % 4 == 0, 1, 0)       # 1 = Summer, 0 = Winter

# history features (look only at earlier Games of the same season)
df = df.sort_values(["Athlete", "Season", "Year"])
df["Prev_Appearances"] = df.groupby(["Athlete", "Season"]).cumcount()
df["Prev_Medals"] = df.groupby(["Athlete", "Season"])["Total"].cumsum() - df["Total"]
df["Gold_Share"] = df["Gold"] / df["Total"]

# label: does the athlete medal again 4 years later?
seen = set(zip(df["Athlete"], df["Year"]))
df["Label"] = [int((a, y + 4) in seen) for a, y in zip(df["Athlete"], df["Year"])]

data = df[df["Year"] <= 2008].copy()      # rows that have a known answer
future = df[df["Year"] >= 2010].copy()    # rows whose next Games are not in the file

print("Rows with known label :", len(data))
print("Rows for future use   :", len(future))
print("Share who medalled again (label = 1): {:.1%}".format(data["Label"].mean()))
data.head()

## 4. Visualization

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 10))

sns.histplot(data["Age"], bins=25, ax=ax[0, 0])
ax[0, 0].set_title("Age of medal winners")

data["Total"].value_counts().sort_index().plot(kind="bar", ax=ax[0, 1])
ax[0, 1].set_title("Medals won per athlete per Games")
ax[0, 1].set_xlabel("Total medals"); ax[0, 1].set_ylabel("Athletes")

df.groupby("Year")["Total"].sum().plot(kind="bar", ax=ax[0, 2])
ax[0, 2].set_title("Total medals in the file, by year")

data["Label"].value_counts().sort_index().plot(kind="bar", ax=ax[1, 0], color=["tab:gray", "tab:green"])
ax[1, 0].set_title("Won a medal again? (0 = no, 1 = yes)")
ax[1, 0].set_xticklabels(["No", "Yes"], rotation=0)

sns.boxplot(x="Label", y="Age", data=data, ax=ax[1, 1])
ax[1, 1].set_title("Age vs medalling again")

data.groupby("Total")["Label"].mean().mul(100).plot(kind="bar", ax=ax[1, 2])
ax[1, 2].set_title("% who medalled again, by medals won")
ax[1, 2].set_ylabel("%")

plt.tight_layout()
plt.show()

## 5. Distributions (Normal vs Uniform)

In [ ]:
age = data["Age"]
print("Mean:", round(age.mean(), 2), " Std:", round(age.std(), 2))
print("Skewness:", round(stats.skew(age), 2), " (0 = symmetric)")

x = np.linspace(age.min(), age.max(), 200)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Normal curve on top of the real Age data
ax[0].hist(age, bins=25, density=True, alpha=0.6, label="Real data")
ax[0].plot(x, stats.norm.pdf(x, age.mean(), age.std()), color="red", label="Normal curve")
ax[0].set_title("Age vs Normal distribution"); ax[0].legend()

# Uniform sample between min and max age (for comparison)
uniform = np.random.uniform(age.min(), age.max(), 5000)
ax[1].hist(uniform, bins=25, density=True, alpha=0.6, color="orange")
ax[1].set_title("Uniform distribution (same range)")
plt.show()

stat, p = stats.normaltest(age)
print("Normality test p-value:", p)
print("Age is close to Normal but slightly right-skewed (a few older athletes).")

## 6. Sampling and the Central Limit Theorem
We treat all athletes' ages as the *population* and draw samples from it.

In [ ]:
population = data["Age"]

sample_no_repl = population.sample(n=30, random_state=42)                 # without replacement
sample_repl    = population.sample(n=30, replace=True, random_state=42)   # with replacement

print("Population mean :", round(population.mean(), 3))
print("Sample mean (no replacement):", round(sample_no_repl.mean(), 3))
print("Sample mean (replacement)   :", round(sample_repl.mean(), 3))

In [ ]:
# Sampling distribution of the mean for different sample sizes
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for i, n in enumerate([10, 30, 50, 100]):
    means = [population.sample(n=n, replace=True).mean() for _ in range(1000)]
    ax[i].hist(means, bins=25)
    ax[i].set_title("n = " + str(n) + "   SE = " + str(round(np.std(means), 3)))
    ax[i].set_xlabel("Sample mean age")
plt.suptitle("Central Limit Theorem: bigger samples give a narrower, more normal curve")
plt.show()

print("Theory: Standard Error = std / sqrt(n)")
for n in [10, 30, 50, 100]:
    print("n =", n, " SE =", round(population.std() / np.sqrt(n), 3))

## 7. Hypothesis testing
Two tests that suit this data:

**Test A (Welch t-test):** is the average age different for athletes who medal again vs those who do not?  
**Test B (Chi-square):** does winning 2+ medals at one Games change the chance of medalling again?

In [ ]:
# Test A
again = data.loc[data["Label"] == 1, "Age"]
not_again = data.loc[data["Label"] == 0, "Age"]
t_stat, p_val = stats.ttest_ind(again, not_again, equal_var=False)
print("TEST A: Age, medal again vs not")
print("  Mean age (again):", round(again.mean(), 2), " (not again):", round(not_again.mean(), 2))
print("  t =", round(t_stat, 3), " p-value =", p_val)
print("  ->", "Reject H0: ages differ significantly" if p_val < 0.05 else "Fail to reject H0: no significant age difference")

# Test B
multi = (data["Total"] >= 2).astype(int)
table = pd.crosstab(multi, data["Label"])
chi2, p2, dof, expected = stats.chi2_contingency(table)
print()
print("TEST B: 2+ medals vs medalling again")
print(table)
print("  chi2 =", round(chi2, 2), " p-value =", p2)
print("  ->", "Reject H0: they are related" if p2 < 0.05 else "Fail to reject H0: no relation")

## 8. Correlation and Covariance

In [ ]:
num_cols = ["Age", "Gold", "Silver", "Bronze", "Total", "Season",
            "Prev_Appearances", "Prev_Medals", "Gold_Share", "Label"]

cov = data[num_cols].cov()
corr = data[num_cols].corr()

fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cov, annot=True, fmt=".2f", cmap="coolwarm", ax=ax[0])
ax[0].set_title("Covariance matrix")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Correlation matrix")
plt.tight_layout()
plt.show()

print("Correlation of every feature with the label:")
print(corr["Label"].drop("Label").sort_values(ascending=False).round(3))

In [ ]:
r, p = stats.pearsonr(data["Total"], data["Label"])
rho, p_s = stats.spearmanr(data["Total"], data["Label"])
print("Pearson  (Total medals vs Label): r =", round(r, 3), " p =", p)
print("Spearman (Total medals vs Label): rho =", round(rho, 3), " p =", p_s)
print("Weak but real link: athletes with more medals are somewhat more likely to medal again.")

## 9. Regression models (all six) predicting the probability

Each model predicts a number between 0 and 1 = **probability of medalling again**.  
(The label is 0/1, so a regression model's output is clipped to the 0-1 range and read as a probability.)  
We also add **Logistic Regression**, the standard model built for probabilities.

**Avoiding over/underfitting**
- Athletes are split so the **same athlete is never in both train and test** (prevents leaking).
- Trees are limited (`max_depth`, `min_samples_leaf`) so they cannot memorise the training data.
- We compare **train vs 5-fold cross-validation vs test** scores. Big gaps = overfitting; all scores low = underfitting.

In [ ]:
features = ["Age", "Gold", "Silver", "Bronze", "Total", "Season",
            "Prev_Appearances", "Prev_Medals", "Gold_Share"]

X = data[features]
y = data["Label"]
groups = data["Athlete"]

# 80/20 split, keeping each athlete in only one side
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

print("Training rows:", len(X_train), " Testing rows:", len(X_test))
print("Athletes in both sets:", len(set(g_train) & set(groups.iloc[test_idx])))

In [ ]:
# (model, columns it uses)
models = {
    "Simple Linear Regression":   (LinearRegression(), ["Total"]),
    "Multiple Linear Regression": (LinearRegression(), features),
    "Ridge Regression":           (make_pipeline(StandardScaler(), Ridge(alpha=10.0)), features),
    "Lasso Regression":           (make_pipeline(StandardScaler(), Lasso(alpha=0.005)), features),
    "Decision Tree":              (DecisionTreeRegressor(max_depth=4, min_samples_leaf=30, random_state=42), features),
    "Random Forest":              (RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=30,
                                                         random_state=42, n_jobs=-1), features),
    "Logistic Regression":        (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), features),
}

def get_prob(model, X_part):
    # Return probability (0-1) from any of the models
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_part)[:, 1]
    return np.clip(model.predict(X_part), 0, 1)

cv = GroupKFold(n_splits=5)
rows = []
fitted = {}

for name, (model, cols) in models.items():
    # 5-fold cross-validation on the training data only
    cv_auc, cv_acc = [], []
    for tr, va in cv.split(X_train, y_train, g_train):
        m = clone(model)
        m.fit(X_train.iloc[tr][cols], y_train.iloc[tr])
        p = get_prob(m, X_train.iloc[va][cols])
        cv_auc.append(roc_auc_score(y_train.iloc[va], p))
        cv_acc.append(accuracy_score(y_train.iloc[va], p >= 0.5))

    # fit on full training set and score train + test
    model.fit(X_train[cols], y_train)
    fitted[name] = (model, cols)
    p_tr = get_prob(model, X_train[cols])
    p_te = get_prob(model, X_test[cols])

    rows.append({
        "Model": name,
        "Train Acc": accuracy_score(y_train, p_tr >= 0.5),
        "CV Acc": np.mean(cv_acc),
        "Test Acc": accuracy_score(y_test, p_te >= 0.5),
        "Train AUC": roc_auc_score(y_train, p_tr),
        "CV AUC": np.mean(cv_auc),
        "Test AUC": roc_auc_score(y_test, p_te),
        "Test MAE": mean_absolute_error(y_test, p_te),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, p_te)),
        "Test R2": r2_score(y_test, p_te),
    })

results = pd.DataFrame(rows).sort_values("CV AUC", ascending=False).reset_index(drop=True)
results.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
results.plot(x="Model", y=["Train AUC", "CV AUC", "Test AUC"], kind="bar", ax=ax[0])
ax[0].set_title("AUC: train vs cross-validation vs test (close bars = no overfitting)")
ax[0].set_ylim(0.4, 1); ax[0].tick_params(axis="x", rotation=35)

results.plot(x="Model", y=["Train Acc", "CV Acc", "Test Acc"], kind="bar", ax=ax[1])
ax[1].axhline(1 - y_train.mean(), color="red", linestyle="--", label="Always-say-No baseline")
ax[1].set_title("Accuracy: train vs cross-validation vs test"); ax[1].legend()
ax[1].set_ylim(0.4, 1); ax[1].tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 10. Select the best model
We pick the model with the best **cross-validated AUC** (computed on training data only, so the test set stays untouched).  
AUC measures how well the model *ranks* athletes by their chance of medalling again, which is what matters for a probability.

In [ ]:
best_name = results.loc[0, "Model"]
best_model, best_cols = fitted[best_name]
print("Best model:", best_name)

p_test = get_prob(best_model, X_test[best_cols])
pred = (p_test >= 0.5).astype(int)

baseline = 1 - y_test.mean()
print()
print("TEST accuracy          : {:.1%}".format(accuracy_score(y_test, pred)))
print("Always-say-No baseline : {:.1%}".format(baseline))
print("TEST ROC-AUC           : {:.3f}".format(roc_auc_score(y_test, p_test)))
print("CV ROC-AUC             : {:.3f}".format(results.loc[0, "CV AUC"]))
print("Train ROC-AUC          : {:.3f}".format(results.loc[0, "Train AUC"]))
print()
print(classification_report(y_test, pred, target_names=["No medal next time", "Medal again"], zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=["No", "Yes"]).plot(cmap="Blues")
plt.title("Confusion matrix: " + best_name)
plt.show()

In [ ]:
gap = results.loc[0, "Train AUC"] - results.loc[0, "Test AUC"]
print("Train AUC - Test AUC gap =", round(gap, 3))
if gap > 0.05:
    print("Warning: possible OVERFITTING")
elif results.loc[0, "Train AUC"] < 0.6:
    print("Scores are low on train AND test: the data holds limited signal (UNDERFIT-looking, but more complex models did not help either).")
else:
    print("Train, CV and test scores agree: no overfitting.")

print()
if accuracy_score(y_test, pred) >= 0.80:
    print("Accuracy is above 80%.")
else:
    print("Accuracy is BELOW 80% ({:.1%}) and close to the always-say-No baseline ({:.1%}).".format(
        accuracy_score(y_test, pred), baseline))
    print("Reason: only {:.0%} of medalists medal again, and the columns in this file (age, medal counts,".format(y.mean()))
    print("past medals) carry little signal about who returns. Higher accuracy would need data such as")
    print("sport/event, country, world rankings or injuries.")

## 11. Save the model as a `.pkl` file

In [ ]:
bundle = {
    "model": best_model,
    "model_name": best_name,
    "features": best_cols,
    "is_classifier": hasattr(best_model, "predict_proba"),
}
with open("medal_probability_model.pkl", "wb") as f:
    pickle.dump(bundle, f)
print("Saved: medal_probability_model.pkl")

try:
    from google.colab import files
    files.download("medal_probability_model.pkl")
except ImportError:
    pass

## 12. Use the saved model: probability of winning a medal at the next Olympics

In [ ]:
def predict_medal_probability(age, gold, silver, bronze, season, prev_appearances, prev_medals,
                              model_file="medal_probability_model.pkl"):
    # season: 1 = Summer, 0 = Winter. Returns probability in percent.
    with open(model_file, "rb") as f:
        b = pickle.load(f)
    total = gold + silver + bronze
    row = {"Age": age, "Gold": gold, "Silver": silver, "Bronze": bronze, "Total": total,
           "Season": season, "Prev_Appearances": prev_appearances, "Prev_Medals": prev_medals,
           "Gold_Share": gold / total if total else 0}
    X_new = pd.DataFrame([row])[b["features"]]
    m = b["model"]
    prob = m.predict_proba(X_new)[0, 1] if b["is_classifier"] else float(np.clip(m.predict(X_new)[0], 0, 1))
    return round(prob * 100, 1)

# Example: 24-year-old Summer athlete with 2 golds and 1 bronze, medalled once before (3 medals)
print("Chance of medalling again: ",
      predict_medal_probability(24, 2, 0, 1, season=1, prev_appearances=1, prev_medals=3), "%")

In [ ]:
# Scores for athletes from the last Games in the file (2010 Winter / 2012 Summer): top 10
future = future.copy()
future["Chance_of_next_medal_%"] = (get_prob(best_model, future[best_cols]) * 100).round(1)
future.sort_values("Chance_of_next_medal_%", ascending=False)[
    ["Athlete", "Year", "Age", "Total", "Prev_Medals", "Chance_of_next_medal_%"]].head(10)